# 02 – IPL Data Cleaning & Exploratory Data Analysis (Phase 3)

**Input:** `data/raw/matches.csv`, `data/raw/deliveries.csv`
**Output:** `data/cleaned/matches_cleaned.csv`, `data/cleaned/deliveries_cleaned.csv`

| Part | What we do |
|---|---|
| A | Import libraries and load the data |
| B | Understand the structure |
| C | Remove duplicates |
| D | Standardise team names |
| E | Fix seasons and dates |
| F | Handle missing values |
| G | Standardise venues and cities |
| H | Check for incorrect values |
| I | Add helpful new columns (feature engineering) |
| J | Exploratory Data Analysis (EDA) |
| K | Export cleaned files |

**Golden rule:** never edit the raw files. Clean a *copy* in Python and save it as a new file. That way you can always start again.

Run the cells one at a time, from top to bottom (**Shift + Enter**).

## Part A – Import libraries and load the data

- `matplotlib.pyplot` (`plt`) is Python's basic charting library. We use it for quick EDA charts; the final charts are made in Power BI.
- `%matplotlib inline` is a Jupyter command that shows charts inside the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
%matplotlib inline

pd.set_option("display.max_columns", None)   # show every column
pd.set_option("display.width", 200)          # use the full screen width

RAW_DIR = "../data/raw/"        # Google Colab: "/content/"
CLEAN_DIR = "../data/cleaned/"  # Google Colab: "/content/"

In [ ]:
# Read both CSV files into DataFrames (tables in memory)
matches_raw = pd.read_csv(os.path.join(RAW_DIR, "matches.csv"))
deliveries_raw = pd.read_csv(os.path.join(RAW_DIR, "deliveries.csv"), low_memory=False)

# .copy() makes an independent copy. We clean the copy and keep the raw data unchanged
matches = matches_raw.copy()
deliveries = deliveries_raw.copy()

print("matches   :", matches.shape)
print("deliveries:", deliveries.shape)

**Expected output:** `matches : (1095, 20)` and `deliveries: (260920, 17)`

## Part B – Understand the structure
You did this in notebook 01. Here is a short recap, because you should always look at the data before changing it.

In [ ]:
matches.info()

**What to notice:**
- `date` is stored as text (`object` or `str`). Python cannot do date maths on text, so we convert it in Part E.
- `season` is text, not a number. Some values look like `2007/08`.
- Non-null counts below 1095 (e.g. `city`, `winner`, `method`) mean those columns have missing values.

In [ ]:
matches.describe()

`describe()` covers only number columns: `id`, `result_margin`, `target_runs` and `target_overs`. A margin of 146 runs is a real record (Mumbai Indians vs Delhi, 2017), not an error.

## Part C – Remove duplicate records
A **duplicate** is a row that is exactly the same as another row. Duplicates make totals too big (runs counted twice).

- `duplicated()` returns True for every repeat row.
- `drop_duplicates()` removes them and keeps the first copy.

In [ ]:
print("Duplicate matches   :", matches.duplicated().sum())
print("Duplicate deliveries:", deliveries.duplicated().sum())

# Primary-key check: each match id must appear only once
print("Duplicate match ids :", matches["id"].duplicated().sum())

matches = matches.drop_duplicates()
deliveries = deliveries.drop_duplicates()
print("After removal ->", matches.shape, deliveries.shape)

**Expected output:** all zeros. This dataset has no duplicates, but you still **must check**. In the viva, say: *"I checked for duplicates using `duplicated().sum()`, found none, and still ran `drop_duplicates()` so the pipeline is safe if new data is added."*

## Part D – Standardise team names
Some franchises changed their name over the years, so the same team appears under two names:

| Old name | Current name |
|---|---|
| Delhi Daredevils | Delhi Capitals (renamed in 2019) |
| Kings XI Punjab | Punjab Kings (renamed in 2021) |
| Royal Challengers Bangalore | Royal Challengers Bengaluru (renamed in 2024) |
| Rising Pune Supergiants | Rising Pune Supergiant (spelling changed in 2017) |

If we don't fix this, "Delhi Capitals" would show only its wins since 2019, not the full history.

**Kept separate on purpose:** *Deccan Chargers* and *Sunrisers Hyderabad* are **different franchises** (different owners), and so are *Gujarat Lions* and *Gujarat Titans*. Only real renames are merged.

We use a Python **dictionary** `{old: new}` and `.replace()`. Team names appear in **8 columns** across the two files, so we loop over all of them.

In [ ]:
team_name_map = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

match_team_cols = ["team1", "team2", "toss_winner", "winner"]
delivery_team_cols = ["batting_team", "bowling_team"]

before = len(set(matches["team1"]) | set(matches["team2"]))

for col in match_team_cols:
    matches[col] = matches[col].replace(team_name_map)
for col in delivery_team_cols:
    deliveries[col] = deliveries[col].replace(team_name_map)

teams = sorted(set(matches["team1"]) | set(matches["team2"]))
print(f"Team names: {before} before -> {len(teams)} after")
for t in teams:
    print(" -", t)

**Expected output:** `Team names: 19 before -> 15 after`, followed by the 15 franchise names.

## Part E – Fix the season and date columns

### E1. Convert `date` from text to a real date
`pd.to_datetime()` turns text like `"2008-04-18"` into a datetime value. After that we can get the year and month, sort by date, and so on. `format="%Y-%m-%d"` tells pandas the exact pattern: 4-digit year, then month, then day.

In [ ]:
matches["date"] = pd.to_datetime(matches["date"], format="%Y-%m-%d")
print(matches["date"].dtype)
print("First match:", matches["date"].min().date(), "| Last match:", matches["date"].max().date())

**Expected output:** `datetime64[ns]` (newer pandas shows `datetime64[us]`; both are fine), first match `2008-04-18`, last match `2024-05-26`.

### E2. Fix `season`
Three seasons are written in a cricket-board style: `2007/08` (that is the 2008 IPL), `2009/10` (the 2010 IPL) and `2020/21` (the 2020 IPL, played in the UAE). Each IPL season is played within one calendar year, so the **year of the match date** is the correct season. `.dt.year` gets the year from a datetime column.

In [ ]:
print("Before:", sorted(matches["season"].unique().tolist()))

matches["season"] = matches["date"].dt.year   # e.g. 2008-04-18 -> 2008

print("After :", sorted(matches["season"].unique().tolist()))
print("Number of seasons:", matches["season"].nunique())

**Expected output:** 17 seasons, `2008` … `2024`, all stored as numbers.

## Part F – Handle missing values

There are 3 ways to deal with a missing value. Choose **based on why it is missing**:

| Strategy | When to use | Example here |
|---|---|---|
| **Fill** with a known value | You can work out the correct value | `city` from `venue`, `method` → "Normal" |
| **Keep** as blank (NaN/NULL) | Blank is the correct answer | `winner` in a no-result match |
| **Drop** the row | Row is useless or corrupt | Not needed in this dataset |

In [ ]:
def missing_report(df, name):
    # Shows only the columns that have at least one missing value
    m = df.isnull().sum()
    m = m[m > 0]
    pct = (m / len(df) * 100).round(2)
    print(f"--- {name} ---")
    print(pd.DataFrame({"missing": m, "percent": pct}), "\n")

missing_report(matches, "matches")
missing_report(deliveries, "deliveries")

**Expected output (matches):** `city 51`, `player_of_match 5`, `winner 5`, `result_margin 19`, `target_runs 3`, `target_overs 3`, `method 1074`.

Let's look at each one.

### F1. `city`: fill from the venue
Which venues have a missing city? (`isna()` = is missing)

In [ ]:
matches.loc[matches['city'].isna(), 'venue'].value_counts()

All 51 are UAE grounds, so we know the city. `fillna()` fills blanks only, and `.map()` looks up each venue in our dictionary.

In [ ]:
venue_to_city = {
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah",
}
matches["city"] = matches["city"].fillna(matches["venue"].map(venue_to_city))
print("Missing city now:", matches["city"].isna().sum())

### F2. `winner`, `player_of_match`, `target_runs`, `target_overs`: keep blank

In [ ]:
matches.loc[matches['winner'].isna(), ['id', 'season', 'team1', 'team2', 'result', 'player_of_match']]

All 5 matches with a missing winner were **washed out by rain** (`result = no result`). There was no winner and no Player of the Match, so blank is the **correct** value. Filling it with a team name would be wrong data. The 3 missing `target_runs` are also no-result matches where the second innings never started.

Later, MySQL will store these blanks as `NULL`, and Power BI will show them as `(Blank)`.

### F3. `result_margin`: blank for ties and no-results

In [ ]:
matches.loc[matches['result_margin'].isna(), 'result'].value_counts()

A **tie** has no winning margin. The match was decided by a Super Over instead. A no-result has no margin either, so these stay blank as well.

### F4. `method`: fill with "Normal"
`method` is "D/L" only when the rain rule (Duckworth-Lewis) was used. A blank means it was a normal match, so we fill it with the text `"Normal"`. This makes filters in Power BI clearer.

In [ ]:
matches["method"] = matches["method"].fillna("Normal")
print(matches["method"].value_counts())

**Expected output:** `Normal 1074`, `D/L 21`.

### F5. Deliveries: blanks are expected
`extras_type`, `player_dismissed`, `dismissal_kind` and `fielder` are blank on **most balls**, because most balls are not extras and not wickets. That is correct, so we leave them as they are. We only check that they are **consistent** (Part H).

## Part G – Standardise venues and cities
The same stadium is written in several ways, e.g. `"Eden Gardens"` and `"Eden Gardens, Kolkata"`, or `"Wankhede Stadium"` and `"Wankhede Stadium, Mumbai"`. A venue chart would show them as different grounds.

**Steps:**
1. Keep only the text **before the first comma** (`.str.split(",").str[0]`).
2. Fix spelling differences and renamed stadiums with a dictionary.

In [ ]:
print("Venue names before:", matches["venue"].nunique())

# Step 1: "Eden Gardens, Kolkata" -> "Eden Gardens"
matches["venue"] = matches["venue"].str.split(",").str[0].str.strip()

# Step 2: same stadium, different (or old) name
venue_name_map = {
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "Feroz Shah Kotla": "Arun Jaitley Stadium",                          # renamed 2019
    "Punjab Cricket Association Stadium": "Punjab Cricket Association IS Bindra Stadium",
    "Sardar Patel Stadium": "Narendra Modi Stadium",                     # Motera, renamed 2021
    "Subrata Roy Sahara Stadium": "Maharashtra Cricket Association Stadium",  # Pune, renamed
    "Zayed Cricket Stadium": "Sheikh Zayed Stadium",                     # Abu Dhabi
}
matches["venue"] = matches["venue"].replace(venue_name_map)

print("Venue names after :", matches["venue"].nunique())

**Expected output:** `58` before → `36` after.

In [ ]:
# Cities: Bangalore was renamed Bengaluru
matches["city"] = matches["city"].replace({"Bangalore": "Bengaluru"})
print("Cities:", matches["city"].nunique())

## Part H – Check for incorrect values
"Incorrect" means a value that is **impossible** according to cricket rules. We test each rule. A result of **0** means no problems.

`~` means NOT, and `.isin([...])` checks whether a value is in a list.

In [ ]:
checks = {
    # --- deliveries ---
    "batsman_runs outside 0-6":           (~deliveries["batsman_runs"].between(0, 6)).sum(),
    "over outside 0-19":                  (~deliveries["over"].between(0, 19)).sum(),
    "total_runs != batsman + extra":      (deliveries["total_runs"] != deliveries["batsman_runs"] + deliveries["extra_runs"]).sum(),
    "is_wicket not 0/1":                  (~deliveries["is_wicket"].isin([0, 1])).sum(),
    "wicket but no player_dismissed":     ((deliveries["is_wicket"] == 1) & deliveries["player_dismissed"].isna()).sum(),
    "player_dismissed but no wicket":     ((deliveries["is_wicket"] == 0) & deliveries["player_dismissed"].notna()).sum(),
    "batting team == bowling team":       (deliveries["batting_team"] == deliveries["bowling_team"]).sum(),
    "match_id not in matches":            (~deliveries["match_id"].isin(matches["id"])).sum(),
    # --- matches ---
    "team1 == team2":                     (matches["team1"] == matches["team2"]).sum(),
    "toss_winner not in this match":      (~((matches["toss_winner"] == matches["team1"]) | (matches["toss_winner"] == matches["team2"]))).sum(),
    "winner not in this match":           (matches["winner"].notna() & (matches["winner"] != matches["team1"]) & (matches["winner"] != matches["team2"])).sum(),
    "toss_decision not bat/field":        (~matches["toss_decision"].isin(["bat", "field"])).sum(),
    "negative result_margin":             (matches["result_margin"] < 0).sum(),
}

problems = pd.Series(checks, name="bad_rows")
print(problems)
print("\nALL CHECKS PASSED" if problems.sum() == 0 else "\nSome checks FAILED - investigate!")

**Expected output:** every count is 0, then `ALL CHECKS PASSED`.

**Possible viva question:** *"`batsman_runs` has a value of 5. Isn't that wrong?"* No. It happens 65 times. 5 runs can come from running 1 plus overthrows that reach the boundary. Always check a strange value against the rules of the game before deleting it.

**Another one:** *"Why does `ball` go up to 11?"* Wides and no-balls must be bowled again, so an over can have more than 6 deliveries.

## Part I – Add helpful new columns (feature engineering)
**Feature engineering** means creating new columns from existing ones so that later analysis is easy. If we calculate these once in Python, the SQL queries and DAX measures stay short.

### I1. New columns in `matches`

In [ ]:
# Did the toss winner also win the match? (1 = yes, 0 = no, blank for no-result)
matches["toss_winner_won"] = np.where(
    matches["winner"].isna(), np.nan,
    (matches["toss_winner"] == matches["winner"]).astype(int)
)

# Which team batted first? If toss winner chose to bat, they batted first; otherwise the other team did
matches["batting_first_team"] = np.where(
    matches["toss_decision"] == "bat",
    matches["toss_winner"],
    np.where(matches["toss_winner"] == matches["team1"], matches["team2"], matches["team1"])
)

# Was it a playoff match? Everything that is not "League"
matches["is_playoff"] = (matches["match_type"] != "League").astype(int)

matches[["id", "season", "toss_winner", "toss_decision", "winner", "toss_winner_won", "batting_first_team", "is_playoff"]].head()

`np.where(condition, value_if_true, value_if_false)` works like Excel's `IF()`, but for a whole column at once.

### I2. New columns in `deliveries`

| New column | Rule | Why we need it |
|---|---|---|
| `season` | copied from matches using `match_id` | filter player stats by season |
| `over_number` | `over + 1` | the dataset counts overs from 0; people count from 1 |
| `phase` | overs 1–6 Powerplay, 7–15 Middle, 16–20 Death | phase-wise analysis |
| `is_super_over` | `inning > 2` | Super Over balls are excluded from career stats |
| `is_four` / `is_six` | `batsman_runs` = 4 / 6 | boundary counts |
| `is_dot_ball` | `total_runs` = 0 | dot-ball % |
| `ball_faced` | 1 unless it was a **wide** | **strike rate** = runs ÷ balls faced × 100 |
| `is_legal_ball` | not a wide and not a no-ball | **overs bowled** = legal balls ÷ 6 |
| `bowler_runs` | total runs minus byes, leg-byes and penalty | **economy**: byes and leg-byes are not the bowler's fault |
| `is_bowler_wicket` | wicket, excluding run outs and retirements | **wickets** credited to the bowler |

In [ ]:
# --- season: look up each match_id in matches (like VLOOKUP in Excel) ---
season_lookup = matches.set_index("id")["season"]          # a Series: match id -> season
deliveries["season"] = deliveries["match_id"].map(season_lookup)

# --- over number and phase of play ---
deliveries["over_number"] = deliveries["over"] + 1
deliveries["phase"] = pd.cut(
    deliveries["over_number"],
    bins=[0, 6, 15, 20],                    # (0-6], (6-15], (15-20]
    labels=["Powerplay", "Middle", "Death"]
)

# --- simple yes/no (1/0) flags ---
deliveries["is_super_over"] = (deliveries["inning"] > 2).astype(int)
deliveries["is_four"] = (deliveries["batsman_runs"] == 4).astype(int)
deliveries["is_six"] = (deliveries["batsman_runs"] == 6).astype(int)
deliveries["is_dot_ball"] = (deliveries["total_runs"] == 0).astype(int)

# --- balls faced and legal balls ---
deliveries["ball_faced"] = (deliveries["extras_type"] != "wides").astype(int)
deliveries["is_legal_ball"] = (~deliveries["extras_type"].isin(["wides", "noballs"])).astype(int)

# --- runs charged to the bowler ---
not_bowler_extras = deliveries["extras_type"].isin(["byes", "legbyes", "penalty"])
deliveries["bowler_runs"] = deliveries["total_runs"] - np.where(not_bowler_extras, deliveries["extra_runs"], 0)

# --- wickets credited to the bowler ---
not_bowler_wicket = ["run out", "retired hurt", "retired out", "obstructing the field"]
deliveries["is_bowler_wicket"] = (
    (deliveries["is_wicket"] == 1) & (~deliveries["dismissal_kind"].isin(not_bowler_wicket))
).astype(int)

print("New deliveries shape:", deliveries.shape)
print(deliveries["phase"].value_counts())

**Expected output:** `(260920, 28)`, i.e. 17 original columns plus 11 new ones, and ball counts for Middle, Powerplay and Death.

**Note:** `.isin()` on a blank cell gives False, so `extras_type != "wides"` is True for normal balls. That is exactly what we want.

## Part J – Exploratory Data Analysis (EDA)
EDA means asking simple questions of the data to find patterns **before** building the dashboard. Each answer below becomes a chart or KPI in Phase 5.

For player stats we **exclude Super Over balls**, as official IPL records do.

In [ ]:
regular = deliveries[deliveries["is_super_over"] == 0]   # filter rows (like WHERE in SQL)

print("Seasons        :", matches["season"].nunique())
print("Matches        :", len(matches))
print("Teams          :", len(teams))
print("Venues         :", matches["venue"].nunique())
print("Players (batted):", regular["batter"].nunique())
print("Total runs     :", f'{regular["total_runs"].sum():,}')
print("Total wickets  :", f'{regular["is_wicket"].sum():,}')
print("Total sixes    :", f'{regular["is_six"].sum():,}')

These are your **KPI cards** for Dashboard Page 1.

### J1. Matches per season

In [ ]:
matches_per_season = matches.groupby("season")["id"].count()
print(matches_per_season)

matches_per_season.plot(kind="bar", figsize=(10, 4), title="Matches per season", color="steelblue")
plt.ylabel("Matches")
plt.show()

`groupby("season")` puts the rows into one group per season, and `["id"].count()` counts the matches in each group. This is the same as `GROUP BY` + `COUNT()` in SQL (Phase 4). Seasons with **10 teams** (2011–2013, and 2022 onwards) have about 74 matches; seasons with 8 teams have about 60.

### J2. Most successful teams (total wins and win %)

In [ ]:
wins = matches["winner"].value_counts()
played = pd.concat([matches["team1"], matches["team2"]]).value_counts()

team_stats = pd.DataFrame({"played": played, "wins": wins}).fillna(0)
team_stats["win_pct"] = (team_stats["wins"] / team_stats["played"] * 100).round(1)
team_stats = team_stats.sort_values("wins", ascending=False)
team_stats

`pd.concat` stacks `team1` and `team2` into one long column, so `value_counts()` gives the number of matches each team played. **Expected:** Mumbai Indians have the most wins (144), but Gujarat Titans have the best win % (62.2%) and Chennai Super Kings the best win % among older teams (58.0%). **Win %** is fairer than total wins, because newer teams have played fewer matches.

### J3. Title winners (who won each Final)

In [ ]:
finals = matches[matches["match_type"] == "Final"].sort_values("season")
champions = finals[["season", "winner"]].set_index("season")
print(champions)
print("\nTitles per team:")
print(champions["winner"].value_counts())

### J4. Does winning the toss help?

In [ ]:
decided = matches[matches["winner"].notna()]                 # ignore no-result matches
toss_win_pct = decided["toss_winner_won"].mean() * 100
print(f"Toss winner also won the match: {toss_win_pct:.1f}% of matches")

print("\nToss decisions per season (%):")
toss_trend = pd.crosstab(matches["season"], matches["toss_decision"], normalize="index").mul(100).round(0)
print(toss_trend)

The mean of a 0/1 column is the **percentage of 1s**, a useful trick. If the toss winner won about 50% of the time, the toss would make no difference. **Expected: 50.8%**, barely above 50%, so the toss hardly affects the result. The crosstab also shows that captains choose to **field first** much more often in recent seasons.

### J5. Top 10 run scorers

In [ ]:
batting = regular.groupby("batter").agg(
    runs=("batsman_runs", "sum"),
    balls=("ball_faced", "sum"),
    fours=("is_four", "sum"),
    sixes=("is_six", "sum"),
)
batting["strike_rate"] = (batting["runs"] / batting["balls"] * 100).round(2)
top_batters = batting.sort_values("runs", ascending=False).head(10)
top_batters

**Expected:** V Kohli at the top with **8004** runs (strike rate 131.97). `.agg()` calculates several summaries at once. Each line reads as `new_column=("source_column", "function")`.

In [ ]:
top_batters["runs"].sort_values().plot(kind="barh", figsize=(8, 5), title="Top 10 run scorers (2008-2024)", color="darkorange")
plt.xlabel("Runs")
plt.show()

### J6. Best strike rate (minimum 500 balls faced)

In [ ]:
qualified = batting[batting["balls"] >= 500]
qualified.sort_values("strike_rate", ascending=False).head(10)

Without a **minimum-balls filter**, a player who faced 2 balls and hit 2 sixes would have a strike rate of 600. Always set a qualifying limit; the SQL version uses `HAVING`.

### J7. Top 10 wicket takers and best economy

In [ ]:
bowling = regular.groupby("bowler").agg(
    wickets=("is_bowler_wicket", "sum"),
    runs_conceded=("bowler_runs", "sum"),
    legal_balls=("is_legal_ball", "sum"),
    dot_balls=("is_dot_ball", "sum"),
)
bowling["overs"] = (bowling["legal_balls"] / 6).round(1)
bowling["economy"] = (bowling["runs_conceded"] / (bowling["legal_balls"] / 6)).round(2)

print("Top 10 wicket takers:")
display(bowling.sort_values("wickets", ascending=False).head(10))

print("Best economy (minimum 1000 legal balls, i.e. ~167 overs):")
display(bowling[bowling["legal_balls"] >= 1000].sort_values("economy").head(10))

### J8. Venues that host the most matches

In [ ]:
venue_stats = matches.groupby("venue").agg(matches=("id", "count")).sort_values("matches", ascending=False)
venue_stats.head(10)

### J9. Bat first or chase? Which wins more often?

In [ ]:
decided = matches[matches["winner"].notna() & (matches["result"] != "tie")]
bat_first_won = (decided["winner"] == decided["batting_first_team"]).mean() * 100
print(f"Batting first won : {bat_first_won:.1f}%")
print(f"Chasing team won  : {100 - bat_first_won:.1f}%")

**Expected:** batting first 45.7%, chasing 54.3%. This is why most captains choose to field first after winning the toss (see J4).

### J10. Average runs per over by phase

In [ ]:
phase_stats = regular.groupby("phase", observed=True).agg(
    runs=("total_runs", "sum"),
    legal_balls=("is_legal_ball", "sum"),
    wickets=("is_wicket", "sum"),
)
phase_stats["run_rate"] = (phase_stats["runs"] / (phase_stats["legal_balls"] / 6)).round(2)
phase_stats

**Expected:** run rate ≈ 7.85 (Powerplay), 7.79 (Middle), **9.96 (Death)**. Teams score fastest in the death overs (16–20), when batters take more risks, and wickets fall most often per ball in that phase too.

### J11. Player of the Match: top 10

In [ ]:
matches['player_of_match'].value_counts().head(10)

## Part K – Export the cleaned datasets
- `index=False` stops pandas from writing its row numbers (0, 1, 2 …) as an extra column.
- `date_format="%Y-%m-%d"` writes dates in the format MySQL expects (`YYYY-MM-DD`).
- Blank values (NaN) are written as empty cells. MySQL will read them as `NULL` in Phase 4.
- `lineterminator="\n"` ends each line the same way on Windows, Mac and Linux, so the MySQL import works the same everywhere.

**Why convert to `Int64` first?** A column with any blank cells becomes a decimal (float) column in pandas, so `140` gets saved as `140.0`. `Int64` (capital **I**) is pandas' *nullable integer* type: it keeps whole numbers **and** allows blanks.

In [ ]:
# Whole-number columns that contain blanks: save them as integers, not 140.0
for col in ["result_margin", "target_runs", "toss_winner_won"]:
    matches[col] = matches[col].astype("Int64")

os.makedirs(CLEAN_DIR, exist_ok=True)      # create the folder if it does not exist

matches_out = os.path.join(CLEAN_DIR, "matches_cleaned.csv")
deliveries_out = os.path.join(CLEAN_DIR, "deliveries_cleaned.csv")

matches.to_csv(matches_out, index=False, date_format="%Y-%m-%d", lineterminator="\n")
deliveries.to_csv(deliveries_out, index=False, lineterminator="\n")

print("Saved:", matches_out, matches.shape)
print("Saved:", deliveries_out, deliveries.shape)

In [ ]:
# Re-read the files to confirm they were saved correctly
check_m = pd.read_csv(matches_out)
check_d = pd.read_csv(deliveries_out)
print(check_m.shape, check_d.shape)
check_m.head(3)

**Expected output:** `matches (1095, 23)` and `deliveries (260920, 28)`.

## ✅ Phase 3 complete. Summary of the cleaning
| Problem found | What we did |
|---|---|
| Duplicates | Checked: 0 found |
| 19 team names for 15 franchises | Mapped 4 old names to the current names |
| Mixed season formats (`2007/08`) | Took the season from the match date's year |
| `date` stored as text | Converted to datetime |
| 51 missing cities | Filled from the venue (Dubai/Sharjah) |
| Missing winner / margin / target | Kept blank: correct for no-result and tie matches |
| Missing `method` | Filled with "Normal" |
| 58 venue spellings | Standardised to 36 venues |
| Impossible values | 13 rule checks, all passed |
| Hard-to-calculate stats | Added 3 match columns and 11 delivery columns |

Next is **Phase 4**: load these cleaned files into MySQL.